# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    create_experiment_run,
    load_commutative_cnn_pretraining_config,
    persist_pretraining_artifacts,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [ ]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
experiment_output_dir = Path("artifacts/pretrained_commutative_cnn")
previous_pretraining_config_path = experiment_output_dir / "config.yaml"
previous_pretrained_encoder_path = None
if previous_pretraining_config_path.exists():
    previous_pretraining_config = load_commutative_cnn_pretraining_config(previous_pretraining_config_path)
    previous_pretrained_encoder_path = previous_pretraining_config.pretrained_encoder_path
    if not previous_pretrained_encoder_path.exists():
        previous_pretrained_encoder_path = None
experiment_run = create_experiment_run(experiment_output_dir, "10C_pretrain_commutative_cnn")
pretrained_encoder_path = Path(experiment_run.run_dir) / f"{experiment_run.experiment_id}_encoder_state.pt"
validation_fraction = 0.15
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(16, 32),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(48, 64),
    temporal_st_kernel_sizes=(5, 3),
    temporal_ts_channels=(32, 48, 64),
    temporal_ts_kernel_sizes=(7, 5, 3),
    spatial_agg_channels=(32, 64),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=64,
    num_prototypes=64,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.25,
    normalization="group",
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=140,
    learning_rate=3e-5,
    weight_decay=1e-3,
    early_stopping_patience=16,
    early_stopping_min_delta=5e-5,
    early_stopping_start_epoch=20,
    early_stopping_monitor="self_probe_loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=3,
    training_plot_dir=str(Path(experiment_run.loss_plot_dir) / "pretraining"),
    training_plot_every_n_epochs=2,
    training_plot_smoothing_window=5,
    scheduler_patience=3,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.0,
    cross_warmup_epochs=0,
    cross_ramp_epochs=0,
    prototype_temperature=0.25,
    prototype_alignment_weight=0.0,
    prototype_warmup_epochs=16,
    prototype_ramp_epochs=36,
    latent_alignment_weight=0.0,
    lambda_align=0.0,
    probe_mask_probability=1.0,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=0.25,
    probe_alpha_frequency=0.10,
    probe_alpha_correlation=0.05,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(
    pretraining_config,
    Path(experiment_run.run_dir) / f"{experiment_run.experiment_id}_config.yaml",
)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Experiment id: {experiment_run.experiment_id}")
print(f"Experiment run folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Per-run commutative CNN pretraining config in {pretraining_config_path}")
print(f"Pretraining loss PDFs: {Path(optimization_config.training_plot_dir).resolve()}")
print(f"Pretrained encoder checkpoint target: {pretrained_encoder_path.resolve()}")
print(f"Warm-start encoder checkpoint: {None if previous_pretrained_encoder_path is None else previous_pretrained_encoder_path.resolve()}")
pretraining_config


In [ ]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


## Output Review

The GroupNorm/no-cross-pressure diagnostic stabilized pretraining: validation self-probe improved monotonically through epoch `70`, with the best checkpoint at the final epoch. That means the prior failure was most likely BatchNorm/cross-pressure instability, not an inherent model/data failure.

This next run is an extended warm-start convergence check. It reloads the latest stable encoder from `artifacts/pretrained_commutative_cnn/config.yaml`, keeps `normalization="group"`, keeps `lambda_cross=0.0` and prototype/latent alignment disabled, and extends training to `140` epochs with patience `16`. The goal is a better pretrained representation before the compound-focused 13C fine-tune, while still avoiding the auxiliary pressures that previously destabilized validation.

In [ ]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
    pretrained_state_path=previous_pretrained_encoder_path,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
latest_pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
print(f"Updated latest commutative CNN pretraining config at {latest_pretraining_config_path}")
pretraining_artifacts = persist_pretraining_artifacts(
    output_dir=experiment_output_dir,
    estimator=model,
    config=pretraining_config,
    experiment_prefix="10C_pretrain_commutative_cnn",
    experiment_id=experiment_run.experiment_id,
    pretrained_encoder_path=pretrained_encoder_path,
    loss_plot_dirs=[optimization_config.training_plot_dir],
    analysis=(
        "Planned extended warm-start run: continue from the stable GroupNorm/no-cross checkpoint and test "
        "whether validation self-probe reaches a plateau with 140 total epochs and patience 16. After the run, "
        "replace this with best epoch, final validation self-probe, and whether the train/validation gap narrowed."
    ),
    next_round_proposal=(
        "If validation self-probe plateaus cleanly, use the resulting checkpoint for 13C and then test "
        "lambda_cross=0.005-0.01 in a separate pretraining run. If the gap keeps shrinking at epoch 140, "
        "extend once more before adding auxiliary pressure."
    ),
)
pretraining_artifacts


In [ ]:
model.pretrain_history_.tail()